# DC-AC Converters (Inverters)

This chapter covers single-phase and three-phase inverters, including six-step and SPWM strategies.

Topics covered:
- Single-phase half-bridge and full-bridge square-wave
- Three-phase six-step operation in 120 degree and 180 degree conduction
- Bipolar and unipolar SPWM with tunable $m_a$ and $m_f$

## Key Theory

For square-wave inverters, dominant fundamental components are:

- Half-bridge output levels $\pm V_{dc}/2$:
$$V_{1,pk}=\frac{2}{\pi}V_{dc}, \quad V_{1,rms}=\frac{V_{1,pk}}{\sqrt{2}}$$

- Full-bridge output levels $\pm V_{dc}$:
$$V_{1,pk}=\frac{4}{\pi}\frac{V_{dc}}{2}=\frac{2V_{dc}}{\pi}$$

For SPWM in linear modulation:
$$V_{1,pk} \approx m_a V_{dc}, \quad m_a\leq 1$$
with frequency ratio $m_f=f_c/f_1$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from power_sim import dc_ac, loads, plotter, utils

In [ ]:
f1 = 50.0
dc_bus = 300.0
time = utils.timebase(fundamental_frequency=f1, cycles=1.0, samples_per_cycle=5000)
omega_t = plotter.to_omega_t(time, f1)

## Single-Phase Square-Wave Inverters

Half-bridge uses one leg and split bus midpoint, while full-bridge uses two legs to double output swing.

In [ ]:
half_bridge = dc_ac.single_phase_half_bridge_square(time, dc_bus=dc_bus, output_frequency=f1)
full_bridge = dc_ac.single_phase_full_bridge_square(time, dc_bus=dc_bus, output_frequency=f1)

print('Half-bridge theory:', half_bridge['theory'])
print('Half-bridge simulated metrics:', half_bridge['metrics'])
print('Full-bridge theory:', full_bridge['theory'])
print('Full-bridge simulated metrics:', full_bridge['metrics'])

In [ ]:
fig, _ = plotter.plot_stacked_waveforms(
    x=omega_t,
    traces=[
        {'y': half_bridge['output_voltage'], 'label': 'Half-bridge output', 'ylabel': 'v_o (V)'},
        {'y': full_bridge['output_voltage'], 'label': 'Full-bridge output', 'ylabel': 'v_o (V)'},
    ],
    xlabel='Electrical angle, omega t (rad)',
    title='Single-Phase Square-Wave Inverter Outputs'
)
plt.show()

## SPWM: Bipolar vs Unipolar

Comparator logic:
- Bipolar SPWM: one comparator controls bridge polarity directly.
- Unipolar SPWM: each leg has its own reference ($+\sin$ and $-\sin$), reducing effective harmonic content around output fundamental.

In [ ]:
m_a = 0.85
f_carrier = 5000.0

bipolar = dc_ac.single_phase_bipolar_spwm(
    time, dc_bus=dc_bus, output_frequency=f1, carrier_frequency=f_carrier, modulation_index=m_a
)
unipolar = dc_ac.single_phase_unipolar_spwm(
    time, dc_bus=dc_bus, output_frequency=f1, carrier_frequency=f_carrier, modulation_index=m_a
)

print('Bipolar SPWM m_a, m_f:', bipolar['m_a'], bipolar['m_f'])
print('Bipolar SPWM theory:', bipolar['theory'])
print('Unipolar SPWM theory:', unipolar['theory'])

In [ ]:
sample_window = slice(0, 1400)
fig, _ = plotter.plot_stacked_waveforms(
    x=omega_t[sample_window],
    traces=[
        {'y': bipolar['reference'][sample_window], 'label': 'Reference', 'ylabel': 'Ref/Carrier'},
        {'y': bipolar['carrier'][sample_window], 'label': 'Carrier', 'ylabel': 'Ref/Carrier'},
        {'y': bipolar['output_voltage'][sample_window], 'label': 'Bipolar output', 'ylabel': 'v_o (V)'},
        {'y': unipolar['output_voltage'][sample_window], 'label': 'Unipolar output', 'ylabel': 'v_o (V)'},
    ],
    xlabel='Electrical angle, omega t (rad)',
    title='SPWM Waveforms (Zoomed)'
)
plt.show()

## Three-Phase Six-Step Inverter

Conduction modes:
- 180 degree mode: each device conducts for half-cycle.
- 120 degree mode: each device conducts for 120 degree with one phase floating each interval.

In [ ]:
inv_180 = dc_ac.three_phase_six_step(time, dc_bus=400.0, output_frequency=f1, conduction_mode=180)
inv_120 = dc_ac.three_phase_six_step(time, dc_bus=400.0, output_frequency=f1, conduction_mode=120)

print('180 degree conduction theory:', inv_180['theory'])
print('120 degree conduction theory:', inv_120['theory'])

In [ ]:
vab_180 = inv_180['line_voltages']['v_ab']
vab_120 = inv_120['line_voltages']['v_ab']

high_L = loads.IdealCurrentLoad(current_level=8.0)
load_wave = high_L.simulate(time, vab_180)

fig, _ = plotter.plot_stacked_waveforms(
    x=omega_t,
    traces=[
        {'y': vab_180, 'label': 'v_ab (180 degree)', 'ylabel': 'Line voltage (V)'},
        {'y': vab_120, 'label': 'v_ab (120 degree)', 'ylabel': 'Line voltage (V)'},
        {'y': load_wave.current, 'label': 'Ideal inductive current', 'ylabel': 'i (A)'},
    ],
    xlabel='Electrical angle, omega t (rad)',
    title='Three-Phase Six-Step Inverter: 180 vs 120 Conduction'
)
plt.show()